# Frozen Lake Example

In [ ]:
import gymnasium as gym
import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

## Registering the Frozen Lake environment

In [ ]:
gym.register(
    id="FrozenLake-v2",
    entry_point="techdays26.frozen_lake.frozen_lake_enhanced:FrozenLakeEnv",
    kwargs={"map_name": "8x8"},
    max_episode_steps=200,
    reward_threshold=0.85,
)


def make_frozen_lake(render_mode="human", show_values=False, slippery=False):
    return gym.make(
        "FrozenLake-v2",
        desc=None,  # generate_random_map(size=8)
        map_name="8x8",
        show_q_labels=show_values,
        is_slippery=slippery,
        success_rate=3 / 4,
        reward_schedule=(1, -1, 0),
        render_mode=render_mode,
    )

## Introducing the Problem

In [ ]:
# https://github.com/johnnycode8/gym_solutions

import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

episodes: int = 2

env = make_frozen_lake()

try:
    env.unwrapped.set_info({
        "Mode": "Manual (Arrow Keys)",
    })

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            # Wait for an arrow key press
            action = get_action_from_keyboard(key_to_action)
            if action is None:
                raise SystemExit  # on Escape

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## The Policy Function
Illustrate how $\pi(s,a)$ works

In [ ]:
from techdays26.frozen_lake.frozen_lake_utils import action_to_arrow

print("Mapping of actions to directions:")
print(action_to_arrow)

### Specify a simple Policy
What do the following policies do? Can you guess?

In [ ]:
# To act, sample from the distribution:
def act(state, π):  # TODO: typing
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)

In [ ]:
def π_1(state, action):
    """π(a|s) → probability"""
    #         ←    ↓    →    ↑
    probs = [0.05, 0.45, 0.45, 0.05]
    return probs[action]

In [ ]:
def π_2(state: int, action: int):
    """π(a|s) → probability"""
    if state % 8 < 7:
        #         ←    ↓    →    ↑
        probs = [0.0, 0.0, 1.0, 0.0]
    else:
        #         ←    ↓    →    ↑
        probs = [0.0, 1.0, 0.0, 0.0]

    return probs[action]

### Running an agent with above policies

In [ ]:
episodes: int = 20
π = π_2  # Callable or "manual"

env = make_frozen_lake()

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            ####################################################################
            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            ####################################################################

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## Value Functions

### Estimating the Value Function $V^\pi(s)$ with Monte Carlo Rollouts
- Why they are called Monte Carlo Rollouts?

#### Using a manual policy or a pre-defined Policy

In [ ]:
episodes: int = 5

π = "manual"

env = make_frozen_lake(show_values=True)

################################################################################
nS = env.observation_space.n
v = np.zeros((nS,))
returns_sum = np.zeros(nS)
returns_count = np.zeros(nS)
gamma = 0.9  # discount factor γ
################################################################################

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        episode = []  # <--------------------------------------------------- NEW

        while not terminated and not truncated:
            env.unwrapped.set_v(v)  # <------------------------------------- NEW
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            ####################################################################
            episode.append((state, reward))
            ####################################################################

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        ########################################################################
        # compute returns backwards and update first-visit states
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = r_t + gamma * G
            if s_t not in visited:
                visited.add(s_t)
                returns_sum[s_t] += G
                returns_count[s_t] += 1
                v[s_t] = returns_sum[s_t] / returns_count[s_t]  # Expected value
        ########################################################################

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

### Applying the Bellman Equation: Estimating the Value Function $V^\pi(s)$ with TD(0)

In [ ]:
episodes: int = 200000
π = π_1

env = make_frozen_lake(show_values=True)

################################################################################
v = np.zeros((env.observation_space.n,))
gamma = 0.9  # discount factor γ
alpha = 0.01  # step size α
################################################################################

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            action = act(state=state, π=π)
            new_state, reward, terminated, truncated, info_dict = env.step(action)

            # ---------- TD(0) policy evaluation update for V^π ----------
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])
            td_error = td_target - v[state]
            v[state] += alpha * td_error
            # ------------------------------------------------------------

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()